# Моделирование. Проведение экспериментов. Подготовка пайплайнов обработки данных и построения модели.

# Импорты и настройки

In [ ]:
# Стандартная библиотека
import os
import sys
import warnings
from pathlib import Path

import joblib

# Работа с данными
import numpy as np
import pandas as pd

# ML
import lightgbm as lgb
import mlflow
import mlflow.lightgbm

from scipy import sparse
from scipy.sparse import csr_matrix

from implicit.als import AlternatingLeastSquares

# Настройки
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 300)
pd.options.display.float_format = "{:.6f}".format

# Пути проекта
CURRENT_DIR = Path.cwd().resolve()

if CURRENT_DIR.name == "notebooks":
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.config import TRAIN_PATH, DATA_DIR, RAW_DIR

MODELS_DIR = PROJECT_ROOT / "models"
MLFLOW_DIR = PROJECT_ROOT / "mlflow"
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"

MODELS_DIR.mkdir(parents=True, exist_ok=True)
MLFLOW_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

# MLflow настройка (ВАЖНО)
MLFLOW_DB_PATH = MLFLOW_DIR / "mlflow.db"
MLFLOW_ARTIFACTS_PATH = MLFLOW_DIR / "artifacts"

MLFLOW_ARTIFACTS_PATH.mkdir(parents=True, exist_ok=True)

mlflow.set_tracking_uri(f"sqlite:///{MLFLOW_DB_PATH}")
mlflow.set_experiment("bank_product_recommendation")

print("MLflow tracking URI:", f"sqlite:///{MLFLOW_DB_PATH}")
print("MLflow artifacts path:", MLFLOW_ARTIFACTS_PATH)

# Нормализация путей
TRAIN_PATH = Path(TRAIN_PATH)
DATA_DIR = Path(DATA_DIR)
RAW_DIR = Path(RAW_DIR)

# Проверка путей
print("\n=== PATHS ===")
print("CURRENT_DIR:", CURRENT_DIR)
print("PROJECT_ROOT:", PROJECT_ROOT)

print("\n=== DATA ===")
print("TRAIN_PATH:", TRAIN_PATH)
print("DATA_DIR:", DATA_DIR)
print("RAW_DIR:", RAW_DIR)

print("\n=== DIRS ===")
print("MODELS_DIR:", MODELS_DIR)
print("MLFLOW_DIR:", MLFLOW_DIR)
print("ARTIFACTS_DIR:", ARTIFACTS_DIR)

print("\n=== EXISTS ===")
print("TRAIN_PATH exists:", TRAIN_PATH.exists())
print("DATA_DIR exists:", DATA_DIR.exists())
print("RAW_DIR exists:", RAW_DIR.exists())
print("MODELS_DIR exists:", MODELS_DIR.exists())
print("MLFLOW_DIR exists:", MLFLOW_DIR.exists())
print("ARTIFACTS_DIR exists:", ARTIFACTS_DIR.exists())

CURRENT_DIR: /home/mle_projects/bank-product-recs/notebooks
PROJECT_ROOT: /home/mle_projects/bank-product-recs
TRAIN_PATH: /home/mle_projects/bank-product-recs/data/raw/train_ver2.csv
DATA_DIR: /home/mle_projects/bank-product-recs/data
RAW_DIR: /home/mle_projects/bank-product-recs/data/raw
MODELS_DIR: /home/mle_projects/bank-product-recs/models
MLFLOW_DIR: /home/mle_projects/bank-product-recs/mlflow
ARTIFACTS_DIR: /home/mle_projects/bank-product-recs/artifacts
TRAIN_PATH exists: True
DATA_DIR exists: True
RAW_DIR exists: True
MODELS_DIR exists: True
MLFLOW_DIR exists: True
ARTIFACTS_DIR exists: True


In [2]:
# функция логирования

mlflow.set_experiment("bank_product_recommendation")

def log_experiment(run_name, metrics, params):
    with mlflow.start_run(run_name=run_name):
        mlflow.log_params(params)
        for k, v in metrics.items():
            mlflow.log_metric(k, float(v))

2026/04/07 09:35:35 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/04/07 09:35:35 INFO mlflow.store.db.utils: Updating database tables
2026/04/07 09:35:37 INFO mlflow.tracking.fluent: Experiment with name 'bank_product_recommendation' does not exist. Creating a new experiment.


# Загрузка данных

In [3]:
df = pd.read_csv(TRAIN_PATH, low_memory=False)
print(df.shape)
df.head()

(13647309, 48)


,fecha_dato,ncodpers,ind_empleado,pais_residencia,sexo,age,fecha_alta,ind_nuevo,antiguedad,indrel,ult_fec_cli_1t,indrel_1mes,tiprel_1mes,indresi,indext,conyuemp,canal_entrada,indfall,tipodom,cod_prov,nomprov,ind_actividad_cliente,renta,segmento,ind_ahor_fin_ult1,ind_aval_fin_ult1,ind_cco_fin_ult1,ind_cder_fin_ult1,ind_cno_fin_ult1,ind_ctju_fin_ult1,ind_ctma_fin_ult1,ind_ctop_fin_ult1,ind_ctpp_fin_ult1,ind_deco_fin_ult1,ind_deme_fin_ult1,ind_dela_fin_ult1,ind_ecue_fin_ult1,ind_fond_fin_ult1,ind_hip_fin_ult1,ind_plan_fin_ult1,ind_pres_fin_ult1,ind_reca_fin_ult1,ind_tjcr_fin_ult1,ind_valo_fin_ult1,ind_viv_fin_ult1,ind_nomina_ult1,ind_nom_pens_ult1,ind_recibo_ult1
0,2015-01-28,1375586,N,ES,H,35,2015-01-12,0.000000,6,1.000000,NaN,1.0,A,S,N,NaN,KHL,N,1.000000,29.000000,MALAGA,1.000000,87218.100000,02 - PARTICULARES,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.000000,0.000000,0
1,2015-01-28,1050611,N,ES,V,23,2012-08-10,0.000000,35,1.000000,NaN,1,I,S,S,NaN,KHE,N,1.000000,13.000000,CIUDAD REAL,0.000000,35548.740000,03 - UNIVERSITARIO,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.000000,0.000000,0
2,2015-01-28,1050612,N,ES,V,23,2012-08-10,0.000000,35,1.000000,NaN,1,I,S,N,NaN,KHE,N,1.000000,13.000000,CIUDAD REAL,0.000000,122179.110000,03 - UNIVERSITARIO,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.000000,0.000000,0
3,2015-01-28,1050613,N,ES,H,22,2012-08-10,0.000000,35,1.000000,NaN,1,I,S,N,NaN,KHD,N,1.000000,50.000000,ZARAGOZA,0.000000,119775.540000,03 - UNIVERSITARIO,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0.000000,0.000000,0
4,2015-01-28,1050614,N,ES,V,23,2012-08-10,0.000000,35,1.000000,NaN,1,A,S,N,NaN,KHE,N,1.000000,50.000000,ZARAGOZA,1.000000,NaN,03 - UNIVERSITARIO,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.000000,0.000000,0


# Базовая предобработка

In [4]:
# Даты
date_cols = ["fecha_dato", "fecha_alta", "ult_fec_cli_1t"]

for col in date_cols:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors="coerce")


# Продуктовые признаки (таргет/фичи)
product_cols = [c for c in df.columns if c.endswith("_ult1")]

for col in product_cols:
    df[col] = (
        pd.to_numeric(df[col], errors="coerce")
        .fillna(0)
        .clip(0, 1)   # защита от мусора
        .astype(np.uint8)
    )


# Числовые признаки
num_cols = [
    "age",
    "antiguedad",
    "renta",
    "ind_nuevo",
    "indrel",
    "tipodom",
    "cod_prov",
    "ind_actividad_cliente"
]

for col in num_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")


# Дополнительная очистка числовых
# возраст
if "age" in df.columns:
    df["age"] = df["age"].clip(18, 100)

# стаж
if "antiguedad" in df.columns:
    df["antiguedad"] = df["antiguedad"].clip(0, 500)

# доход (очень важный признак)
if "renta" in df.columns:
    df["renta"] = df["renta"].clip(0, df["renta"].quantile(0.99))
    df["renta_log"] = np.log1p(df["renta"])


# Категориальные признаки
cat_cols = [
    "ind_empleado",
    "pais_residencia",
    "sexo",
    "indrel_1mes",
    "tiprel_1mes",
    "indresi",
    "indext",
    "canal_entrada",
    "indfall",
    "nomprov",
    "segmento"
]

for col in cat_cols:
    if col in df.columns:
        df[col] = df[col].astype("category")


# Сортировка по времени
df = df.sort_values(["ncodpers", "fecha_dato"]).reset_index(drop=True)

# Формирование таргета

In [5]:
# сортировка по клиенту и времени
df = df.sort_values(["ncodpers", "fecha_dato"]).copy()

# предыдущий продуктовый портфель клиента
prev_products = df.groupby("ncodpers")[product_cols].shift(1)

# признаки предыдущего состояния
prev_product_cols = [f"prev_{col}" for col in product_cols]
prev_products_for_features = prev_products.fillna(0).astype(np.uint8).copy()
prev_products_for_features.columns = prev_product_cols

# таргет: только новые продукты
target_df = (df[product_cols] - prev_products.fillna(0)).clip(lower=0).astype(np.uint8)
target_cols = [f"target_{col}" for col in product_cols]
target_df.columns = target_cols

# объединяем
df = pd.concat([df, prev_products_for_features, target_df], axis=1)

# удаляем первое наблюдение клиента, у которого нет истории
df["row_num_per_client"] = df.groupby("ncodpers").cumcount()
df = df[df["row_num_per_client"] > 0].copy()

print(df.shape)
print("Количество target-признаков:", len(target_cols))
print("Количество prev-признаков:", len(prev_product_cols))
print("NaN в таргете:", df[target_cols].isna().sum().sum())

(12690664, 98)
Количество target-признаков: 24
Количество prev-признаков: 24
NaN в таргете: 0


In [6]:
# распределение таргета
df["new_products"] = df[target_cols].sum(axis=1)

print(df["new_products"].describe())
print(df["new_products"].value_counts(normalize=True).sort_index().head(10))

count   12690664.000000
mean           0.044388
std            0.254697
min            0.000000
25%            0.000000
50%            0.000000
75%            0.000000
max            7.000000
Name: new_products, dtype: float64
new_products
0   0.964706
1   0.028124
2   0.005441
3   0.001544
4   0.000173
5   0.000010
6   0.000000
7   0.000000
Name: proportion, dtype: float64


In [7]:
# самые частые новые продукты
display(df[target_cols].sum().sort_values(ascending=False).head(10))

target_ind_recibo_ult1      153205
target_ind_nom_pens_ult1     84768
target_ind_nomina_ult1       73800
target_ind_cco_fin_ult1      69997
target_ind_tjcr_fin_ult1     69311
target_ind_cno_fin_ult1      37187
target_ind_ecue_fin_ult1     26378
target_ind_dela_fin_ult1     12707
target_ind_reca_fin_ult1      9238
target_ind_ctma_fin_ult1      7002
dtype: uint64

# Feature engineering

In [8]:
# агрегированные признаки
df["num_products"] = df[product_cols].sum(axis=1).astype(np.uint8)

df["tenure_days"] = (df["fecha_dato"] - df["fecha_alta"]).dt.days
df["tenure_days"] = df["tenure_days"].clip(lower=0)
df["tenure_days"] = df["tenure_days"].fillna(df["tenure_days"].median())

df["month"] = df["fecha_dato"].dt.month.astype(np.uint8)

# списки признаков
num_features = [
    "age",
    "antiguedad",
    "renta",
    "renta_log",
    "ind_nuevo",
    "indrel",
    "tipodom",
    "cod_prov",
    "ind_actividad_cliente",
    "num_products",
    "tenure_days",
    "month",
]

cat_features = [
    "ind_empleado",
    "pais_residencia",
    "sexo",
    "indrel_1mes",
    "tiprel_1mes",
    "indresi",
    "indext",
    "canal_entrada",
    "indfall",
    "nomprov",
    "segmento",
]

num_features = [c for c in num_features if c in df.columns]
cat_features = [c for c in cat_features if c in df.columns]

# используем только prev_* как продуктовые признаки
feature_cols = num_features + cat_features + prev_product_cols

print("Количество числовых признаков:", len(num_features))
print("Количество категориальных признаков:", len(cat_features))
print("Количество prev-признаков:", len(prev_product_cols))
print("Итого признаков:", len(feature_cols))

# проверка на leakage
leak_cols = [c for c in feature_cols if c in product_cols]
print("Leakage product_cols in feature_cols:", leak_cols)

Количество числовых признаков: 12
Количество категориальных признаков: 11
Количество prev-признаков: 24
Итого признаков: 47
Leakage product_cols in feature_cols: []


## Обработка категорий

### Для LightGBM можно закодировать категории через category.

In [9]:
# 1. Числовые признаки
num_features = [
    "age",
    "antiguedad",
    "renta",
    "renta_log",
    "ind_nuevo",
    "indrel",
    "tipodom",
    "cod_prov",
    "ind_actividad_cliente",
    "num_products",
    "tenure_days",
    "month",
]

num_features = [c for c in num_features if c in df.columns]

for col in num_features:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# базовая очистка
if "age" in df.columns:
    df["age"] = df["age"].clip(18, 100)

if "antiguedad" in df.columns:
    df["antiguedad"] = df["antiguedad"].clip(0, 500)

if "tenure_days" in df.columns:
    df["tenure_days"] = df["tenure_days"].clip(lower=0)

if "renta" in df.columns:
    upper_renta = df["renta"].quantile(0.99)
    df["renta"] = df["renta"].clip(lower=0, upper=upper_renta)

if "renta_log" not in df.columns and "renta" in df.columns:
    df["renta_log"] = np.log1p(df["renta"])

# медианная импутация
for col in num_features:
    if col in df.columns:
        df[col] = df[col].fillna(df[col].median())


# 2. Категориальные признаки
cat_features = [
    "ind_empleado",
    "pais_residencia",
    "sexo",
    "indrel_1mes",
    "tiprel_1mes",
    "indresi",
    "indext",
    "canal_entrada",
    "indfall",
    "nomprov",
    "segmento",
]

cat_features = [c for c in cat_features if c in df.columns]

for col in cat_features:
    df[col] = df[col].astype(str).replace("nan", "UNKNOWN").astype("category")


# 3. Финальный список признаков

feature_cols = num_features + cat_features + prev_product_cols

print("Количество числовых признаков:", len(num_features))
print("Количество категориальных признаков:", len(cat_features))
print("Количество prev-признаков:", len(prev_product_cols))
print("Итого признаков:", len(feature_cols))

print("\nПример числовых признаков:")
print(num_features[:10])

print("\nПример категориальных признаков:")
print(cat_features[:10])

print("\nПример prev-признаков:")
print(prev_product_cols[:5])

Количество числовых признаков: 12
Количество категориальных признаков: 11
Количество prev-признаков: 24
Итого признаков: 47

Пример числовых признаков:
['age', 'antiguedad', 'renta', 'renta_log', 'ind_nuevo', 'indrel', 'tipodom', 'cod_prov', 'ind_actividad_cliente', 'num_products']

Пример категориальных признаков:
['ind_empleado', 'pais_residencia', 'sexo', 'indrel_1mes', 'tiprel_1mes', 'indresi', 'indext', 'canal_entrada', 'indfall', 'nomprov']

Пример prev-признаков:
['prev_ind_ahor_fin_ult1', 'prev_ind_aval_fin_ult1', 'prev_ind_cco_fin_ult1', 'prev_ind_cder_fin_ult1', 'prev_ind_cno_fin_ult1']


In [10]:
# функция оптимизации памяти
def reduce_memory_usage(df: pd.DataFrame) -> pd.DataFrame:
    start_mem = df.memory_usage(deep=True).sum() / 1024**2
    print(f"Memory before: {start_mem:.2f} MB")

    for col in df.columns:
        col_type = df[col].dtype

        # пропускаем даты
        if pd.api.types.is_datetime64_any_dtype(col):
            continue

        # числовые
        if pd.api.types.is_numeric_dtype(col):
            c_min = df[col].min()
            c_max = df[col].max()

            if pd.api.types.is_integer_dtype(col):
                if c_min >= 0:
                    if c_max < 255:
                        df[col] = df[col].astype(np.uint8)
                    elif c_max < 65535:
                        df[col] = df[col].astype(np.uint16)
                    else:
                        df[col] = df[col].astype(np.uint32)
                else:
                    df[col] = df[col].astype(np.int32)

            else:
                df[col] = df[col].astype(np.float32)

        # строки → category
        elif df[col].dtype == "object":
            num_unique = df[col].nunique()
            num_total = len(df[col])

            if num_unique / num_total < 0.5:
                df[col] = df[col].astype("category")

    end_mem = df.memory_usage(deep=True).sum() / 1024**2
    print(f"Memory after: {end_mem:.2f} MB")
    print(f"Reduced by: {(start_mem - end_mem) / start_mem * 100:.1f}%")

    return df

In [11]:
# оптимизируем память
df = reduce_memory_usage(df)

Memory before: 3074.17 MB
Memory after: 2698.95 MB
Reduced by: 12.2%


## Time-based split

In [12]:
train_df = df[df["fecha_dato"] < "2016-05-28"].copy()
valid_df = df[df["fecha_dato"] == "2016-05-28"].copy()

train_df["new_products"] = train_df[target_cols].sum(axis=1)
valid_df["new_products"] = valid_df[target_cols].sum(axis=1)

print("train shape:", train_df.shape)
print("valid shape:", valid_df.shape)
print("train share with purchases:", (train_df["new_products"] > 0).mean())
print("valid share with purchases:", (valid_df["new_products"] > 0).mean())

train shape: (11763904, 102)
valid shape: (926760, 102)
train share with purchases: 0.03570132840254392
valid share with purchases: 0.030122145970909404


### Почему разбиваем именно так?

* Это последний месяц в train-данных, 2016-05-28 — это последняя дата этого месяца.

* Мы моделируем будущее. Ближайшая дата к “будущему”

## Подготовка матриц признаков

In [13]:
X_train = train_df[feature_cols].copy()
X_valid = valid_df[feature_cols].copy()

for col in cat_features:
    X_train[col] = X_train[col].fillna("UNKNOWN").astype(str).astype("category")
    X_valid[col] = X_valid[col].fillna("UNKNOWN").astype(str).astype("category")

for col in num_features:
    X_train[col] = pd.to_numeric(X_train[col], errors="coerce")
    X_valid[col] = pd.to_numeric(X_valid[col], errors="coerce")

    median_value = X_train[col].median()
    X_train[col] = X_train[col].fillna(median_value)
    X_valid[col] = X_valid[col].fillna(median_value)

print(X_train.shape, X_valid.shape)

(11763904, 47) (926760, 47)


## Готовим матрицу user-item для ALS

In [14]:
MATRIX_PATH = MODELS_DIR / "user_item_matrix.npz"
USER2IDX_PATH = MODELS_DIR / "user2idx.pkl"
ITEM2IDX_PATH = MODELS_DIR / "item2idx.pkl"
IDX2ITEM_PATH = MODELS_DIR / "idx2item.pkl"

if (
    MATRIX_PATH.exists()
    and USER2IDX_PATH.exists()
    and ITEM2IDX_PATH.exists()
    and IDX2ITEM_PATH.exists()
):
    print("Loading cached user-item matrix and mappings...")

    user_item_matrix = sparse.load_npz(str(MATRIX_PATH))
    user2idx = joblib.load(USER2IDX_PATH)
    item2idx = joblib.load(ITEM2IDX_PATH)
    idx2item = joblib.load(IDX2ITEM_PATH)

else:
    print("Building user-item matrix from scratch...")

    train_users = train_df["ncodpers"].drop_duplicates().tolist()
    user2idx = {u: i for i, u in enumerate(train_users)}
    item2idx = {p: i for i, p in enumerate(product_cols)}
    idx2item = {i: p for p, i in item2idx.items()}

    prev_cols = [f"prev_{p}" for p in product_cols]

    missing_prev_cols = [c for c in prev_cols if c not in train_df.columns]
    if missing_prev_cols:
        raise ValueError(f"Missing prev columns: {missing_prev_cols[:5]}")

    interactions = train_df[["ncodpers"] + prev_cols].melt(
        id_vars="ncodpers",
        value_vars=prev_cols,
        var_name="prev_product",
        value_name="has_product",
    )

    interactions = interactions[interactions["has_product"] == 1].copy()
    interactions["product"] = interactions["prev_product"].str.replace("prev_", "", regex=False)

    interactions = interactions.drop_duplicates(subset=["ncodpers", "product"]).copy()

    interactions["user_idx"] = interactions["ncodpers"].map(user2idx)
    interactions["item_idx"] = interactions["product"].map(item2idx)

    interactions = interactions.dropna(subset=["user_idx", "item_idx"]).copy()
    interactions["user_idx"] = interactions["user_idx"].astype(int)
    interactions["item_idx"] = interactions["item_idx"].astype(int)

    user_item_matrix = sparse.csr_matrix(
        (
            np.ones(len(interactions), dtype=np.float32),
            (interactions["user_idx"].values, interactions["item_idx"].values),
        ),
        shape=(len(user2idx), len(item2idx)),
    )

    sparse.save_npz(str(MATRIX_PATH), user_item_matrix)
    joblib.dump(user2idx, USER2IDX_PATH)
    joblib.dump(item2idx, ITEM2IDX_PATH)
    joblib.dump(idx2item, IDX2ITEM_PATH)

    print("Saved matrix and mappings to disk.")

print("MATRIX_PATH:", MATRIX_PATH)
print("USER2IDX_PATH:", USER2IDX_PATH)
print("ITEM2IDX_PATH:", ITEM2IDX_PATH)
print("IDX2ITEM_PATH:", IDX2ITEM_PATH)
print("user_item_matrix shape:", user_item_matrix.shape)
print("nnz:", user_item_matrix.nnz)
print("density:", user_item_matrix.nnz / (user_item_matrix.shape[0] * user_item_matrix.shape[1]))

Building user-item matrix from scratch...
Saved matrix and mappings to disk.
MATRIX_PATH: /home/mle_projects/bank-product-recs/models/user_item_matrix.npz
USER2IDX_PATH: /home/mle_projects/bank-product-recs/models/user2idx.pkl
ITEM2IDX_PATH: /home/mle_projects/bank-product-recs/models/item2idx.pkl
IDX2ITEM_PATH: /home/mle_projects/bank-product-recs/models/idx2item.pkl
user_item_matrix shape: (941815, 24)
nnz: 1456770
density: 0.06444869746181575


# очищаем модели

for path in [
    "models/als_model.pkl",
    "models/user_item_matrix.npz",
    "models/user2idx.pkl",
    "models/item2idx.pkl",
    "models/idx2item.pkl",
]:
    if os.path.exists(path):
        os.remove(path)
        print("Deleted:", path)

# Обучене модели 

## Обучаем ALS

In [15]:
ALS_PATH = os.path.join(MODELS_DIR, "als_model.pkl")

if os.path.exists(ALS_PATH):
    print("Loading cached ALS model...")
    als_model = joblib.load(ALS_PATH)
else:
    print("Training ALS model from scratch...")

    als_model = AlternatingLeastSquares(
        factors=64, regularization=0.01, iterations=20, random_state=42
    )

    alpha = 20.0
    als_train_matrix = (user_item_matrix * alpha).T.tocsr()

    print("ALS train matrix shape:", als_train_matrix.shape)
    print("Expected shape: (n_items, n_users)")

    als_model.fit(user_item_matrix)

    joblib.dump(als_model, ALS_PATH)
    print("ALS model saved:", ALS_PATH)

Training ALS model from scratch...
ALS train matrix shape: (24, 941815)
Expected shape: (n_items, n_users)


  0%|          | 0/20 [00:00<?, ?it/s]

ALS model saved: /home/mle_projects/bank-product-recs/models/als_model.pkl


## Получаем ALS-кандидатов для пользователей validation

In [16]:
def get_existing_products_from_prev(row, product_cols):
    existing = []
    for p in product_cols:
        prev_p = f"prev_{p}"
        if prev_p in row.index and row[prev_p] == 1:
            existing.append(p)
    return existing


def recommend_als_candidates(
    user_id,
    row,
    als_model,
    user2idx,
    user_item_matrix,
    idx2item,
    product_cols,
    n_candidates=20,
):
    existing_products = set(get_existing_products_from_prev(row, product_cols))

    if user_id not in user2idx:
        return []

    user_idx = user2idx[user_id]

    user_items = user_item_matrix[user_idx]
    if user_items.nnz == 0:
        return []

    item_ids, scores = als_model.recommend(
        userid=user_idx,
        user_items=user_items,
        N=min(n_candidates + len(existing_products), len(idx2item)),
        filter_already_liked_items=False,
    )

    recs = []
    for item_idx, score in zip(item_ids, scores):
        item_idx = int(item_idx)

        if item_idx not in idx2item:
            print(f"Warning: item_idx {item_idx} not found in idx2item")
            continue

        p = idx2item[item_idx]

        if p not in existing_products:
            recs.append((p, float(score)))

        if len(recs) >= n_candidates:
            break

    return recs

In [17]:
# Проверка после обучения
sample_row = valid_df.iloc[0]
sample_user = sample_row["ncodpers"]

print("sample_user:", sample_user)
print("mapped user_idx:", user2idx.get(sample_user))
print("n_items:", len(idx2item))
print("idx2item keys sample:", list(idx2item.keys())[:10])

als_candidates = recommend_als_candidates(
    user_id=sample_user,
    row=sample_row,
    als_model=als_model,
    user2idx=user2idx,
    user_item_matrix=user_item_matrix,
    idx2item=idx2item,
    product_cols=product_cols,
    n_candidates=10,
)

print(als_candidates)

sample_user: 15889
mapped user_idx: 0
n_items: 24
idx2item keys sample: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
[('ind_tjcr_fin_ult1', 0.9993175268173218), ('ind_reca_fin_ult1', 0.002942785620689392), ('ind_deme_fin_ult1', 0.0021958127617836), ('ind_recibo_ult1', 0.001716017723083496), ('ind_hip_fin_ult1', 0.0016259625554084778), ('ind_fond_fin_ult1', 0.0015603452920913696), ('ind_ecue_fin_ult1', 0.0015079230070114136), ('ind_viv_fin_ult1', 0.001442648470401764), ('ind_ctop_fin_ult1', 0.001415736973285675), ('ind_dela_fin_ult1', 0.001165047287940979)]


## Baseline Top Popular

### Считаем самые популярные новые продукты на train, добавляем fallback-кандидаты из Top Popular

In [18]:
# Popular products (baseline + fallback)

product_popularity = train_df[target_cols].mean()
product_popularity.index = [c.replace("target_", "") for c in product_popularity.index]

product_prev_popularity = train_df[[f"prev_{p}" for p in product_cols]].mean()
product_prev_popularity.index = [
    c.replace("prev_", "") for c in product_prev_popularity.index
]

product_popularity_df = pd.DataFrame({"product": product_cols})

product_popularity_df["popularity_new"] = product_popularity_df["product"].map(
    product_popularity
)
product_popularity_df["popularity_existing"] = product_popularity_df["product"].map(
    product_prev_popularity
)

product_popularity_df = product_popularity_df.sort_values(
    "popularity_new", ascending=False
).reset_index(drop=True)

# единый источник популярных продуктов
popular_products = product_popularity_df["product"].tolist()

# мапы для фичей
product_popularity_map = dict(
    zip(product_popularity_df["product"], product_popularity_df["popularity_new"])
)
product_prev_popularity_map = dict(
    zip(product_popularity_df["product"], product_popularity_df["popularity_existing"])
)


def get_top_popular_candidates(row, product_cols, popular_products, n_candidates=20):
    existing_products = set(get_existing_products_from_prev(row, product_cols))
    candidates = [p for p in popular_products if p not in existing_products]
    return [(p, 0.0) for p in candidates[:n_candidates]]


product_popularity_df.head()

,product,popularity_new,popularity_existing
0,ind_recibo_ult1,0.012159,0.129241
1,ind_nom_pens_ult1,0.006737,0.060056
2,ind_nomina_ult1,0.005807,0.055376
3,ind_cco_fin_ult1,0.005620,0.664876
4,ind_tjcr_fin_ult1,0.005531,0.045575


## Метрики

In [19]:
def precision_at_k(actual, predicted, k=5):
    actual_set = set(actual)
    predicted = list(dict.fromkeys(predicted[:k]))
    return len(actual_set & set(predicted)) / k if k > 0 else 0.0


def recall_at_k(actual, predicted, k=5):
    actual_set = set(actual)
    predicted = list(dict.fromkeys(predicted[:k]))
    return (
        len(actual_set & set(predicted)) / len(actual_set)
        if len(actual_set) > 0
        else 0.0
    )


def apk(actual, predicted, k=5):
    actual_set = set(actual)
    predicted = predicted[:k]

    score = 0.0
    hits = 0.0
    seen = set()

    for i, p in enumerate(predicted):
        if p in actual_set and p not in seen:
            hits += 1.0
            score += hits / (i + 1.0)
        seen.add(p)

    return score / min(len(actual_set), k) if len(actual_set) > 0 else 0.0


def mean_metrics(actual_list, predicted_list, k=5):
    precisions = [precision_at_k(a, p, k) for a, p in zip(actual_list, predicted_list)]
    recalls = [recall_at_k(a, p, k) for a, p in zip(actual_list, predicted_list)]
    maps = [apk(a, p, k) for a, p in zip(actual_list, predicted_list)]

    return {
        f"precision_at_{k}": float(np.mean(precisions)),
        f"recall_at_{k}": float(np.mean(recalls)),
        f"map_at_{k}": float(np.mean(maps)),
    }

## Подготовка фактических ответов для valid

In [20]:
# Actual для validation
def extract_actual_products(row, target_cols):
    return [c.replace("target_", "") for c in target_cols if row[c] == 1]


def get_existing_products_from_prev(row, product_cols):
    return [p for p in product_cols if row.get(f"prev_{p}", 0) == 1]


def top_popular_recommendations(row, top_popular_products, product_cols, k=5):
    existing_products = set(get_existing_products_from_prev(row, product_cols))
    preds = [p for p in top_popular_products if p not in existing_products]
    return preds[:k]


valid_actual = (
    valid_df[target_cols]
    .apply(lambda row: extract_actual_products(row, target_cols), axis=1)
    .tolist()
)

print("valid rows:", len(valid_actual))
print("non-empty actual:", sum(len(x) > 0 for x in valid_actual))
print(
    "share non-empty actual:",
    round(sum(len(x) > 0 for x in valid_actual) / len(valid_actual), 4),
)

valid rows: 926760
non-empty actual: 27916
share non-empty actual: 0.0301


## Baseline Top Popular

In [21]:
K = 5

popular_targets = train_df[target_cols].sum().sort_values(ascending=False)
top_popular_products = [
    c.replace("target_", "") for c in popular_targets.index.tolist()
]

valid_pred_top_pop = [
    top_popular_recommendations(row, top_popular_products, product_cols, k=K)
    for _, row in valid_df.iterrows()
]

baseline_metrics = mean_metrics(valid_actual, valid_pred_top_pop, k=K)
print("Baseline metrics (all):", baseline_metrics)

# логируем
log_experiment(
    "top_popular",
    baseline_metrics,
    {"model_type": "baseline"}
)

mask_non_empty = [len(x) > 0 for x in valid_actual]
valid_actual_pos = [a for a, m in zip(valid_actual, mask_non_empty) if m]
valid_pred_top_pop_pos = [p for p, m in zip(valid_pred_top_pop, mask_non_empty) if m]

print(
    "Baseline metrics (positive rows only):",
    mean_metrics(valid_actual_pos, valid_pred_top_pop_pos, k=K),
)

Baseline metrics (all): {'precision_at_5': 0.0069791531788165224, 'recall_at_5': 0.02664050023738616, 'map_at_5': 0.01895304429769663}
Baseline metrics (positive rows only): {'precision_at_5': 0.23169508525576735, 'recall_at_5': 0.8844157472417251, 'map_at_5': 0.6292063094044037}


## Обучение LightGBM one-vs-rest

### По одной модели на каждый продукт.

In [22]:
models = {}
valid_scores = pd.DataFrame(index=valid_df.index)

params = {
    "objective": "binary",
    "metric": ["binary_logloss", "auc"],
    "learning_rate": 0.05,
    "num_leaves": 31,
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq": 1,
    "verbosity": -1,
    "seed": 42,
}

MIN_POSITIVES = 300
MODEL_DIR = "models/ovr"
os.makedirs(MODEL_DIR, exist_ok=True)

for i, target_col in enumerate(target_cols, start=1):
    y_train = train_df[target_col].astype(int)
    y_valid = valid_df[target_col].astype(int)

    n_pos_train = int(y_train.sum())
    n_pos_valid = int(y_valid.sum())

    print(
        f"[{i}/{len(target_cols)}] {target_col} | "
        f"train positives={n_pos_train}, valid positives={n_pos_valid}"
    )

    if n_pos_train < MIN_POSITIVES:
        print(f"  -> skip: too few positives in train (< {MIN_POSITIVES})")
        valid_scores[target_col] = 0.0
        continue

    if y_train.nunique() < 2:
        print("  -> skip: only one class in train")
        valid_scores[target_col] = 0.0
        continue

    train_dataset = lgb.Dataset(
        X_train, label=y_train, categorical_feature=cat_features, free_raw_data=False
    )

    valid_dataset = lgb.Dataset(
        X_valid, label=y_valid, categorical_feature=cat_features, free_raw_data=False
    )

    pos_weight = (len(y_train) - y_train.sum()) / max(y_train.sum(), 1)

    params_local = params.copy()
    params_local["scale_pos_weight"] = pos_weight

    model = lgb.train(
        params_local,
        train_dataset,
        valid_sets=[valid_dataset],
        num_boost_round=300,
        callbacks=[lgb.early_stopping(30), lgb.log_evaluation(50)],
    )

    models[target_col] = model
    valid_scores[target_col] = model.predict(
        X_valid, num_iteration=model.best_iteration
    )

    joblib.dump(model, os.path.join(MODEL_DIR, f"{target_col}.pkl"))

print(f"\nОбучено моделей: {len(models)} из {len(target_cols)}")
print("Форма valid_scores:", valid_scores.shape)

[1/24] target_ind_ahor_fin_ult1 | train positives=1, valid positives=1
  -> skip: too few positives in train (< 300)
[2/24] target_ind_aval_fin_ult1 | train positives=4, valid positives=0
  -> skip: too few positives in train (< 300)
[3/24] target_ind_cco_fin_ult1 | train positives=66119, valid positives=3878
Training until validation scores don't improve for 30 rounds
[50]	valid_0's binary_logloss: 0.0614266	valid_0's auc: 0.997062
[100]	valid_0's binary_logloss: 0.0547458	valid_0's auc: 0.997434
[150]	valid_0's binary_logloss: 0.0505745	valid_0's auc: 0.997594
[200]	valid_0's binary_logloss: 0.0474387	valid_0's auc: 0.997704
[250]	valid_0's binary_logloss: 0.0451928	valid_0's auc: 0.99777
[300]	valid_0's binary_logloss: 0.0434988	valid_0's auc: 0.997835
Did not meet early stopping. Best iteration is:
[300]	valid_0's binary_logloss: 0.0434988	valid_0's auc: 0.997835
[4/24] target_ind_cder_fin_ult1 | train positives=131, valid positives=5
  -> skip: too few positives in train (< 300)
[

KeyboardInterrupt: 

## Формирование рекомендаций модели

### Важно не рекомендовать уже имеющиеся у клиента продукты.

In [ ]:
def get_model_recommendations(
    valid_df, score_df, product_cols, top_k=5, top_popular_products=None
):
    recommendations = []

    for idx in valid_df.index:
        valid_row = valid_df.loc[idx]
        score_row = score_df.loc[idx]

        existing_products = {
            p for p in product_cols if valid_row.get(f"prev_{p}", 0) == 1
        }

        scores = {}
        for target_col in score_df.columns:
            product = target_col.replace("target_", "")
            score = score_row.get(target_col, 0.0)

            if pd.isna(score):
                score = 0.0

            if product not in existing_products:
                scores[product] = float(score)

        ranked_products = sorted(scores.items(), key=lambda x: x[1], reverse=True)
        recs = [p for p, _ in ranked_products[:top_k]]

        if top_popular_products is not None and len(recs) < top_k:
            for p in top_popular_products:
                if p not in recs and p not in existing_products:
                    recs.append(p)
                if len(recs) == top_k:
                    break

        recommendations.append(recs)

    return recommendations

## Оценка LightGBM

In [ ]:
valid_pred_lgbm = get_model_recommendations(
    valid_df=valid_df,
    score_df=valid_scores,
    product_cols=product_cols,
    top_k=5,
    top_popular_products=top_popular_products,
)

# all rows
lgbm_metrics_all = mean_metrics(valid_actual, valid_pred_lgbm, k=5)
print("LightGBM one-vs-rest metrics (all):", lgbm_metrics_all)

# positive rows
mask_non_empty = [len(x) > 0 for x in valid_actual]
valid_actual_pos = [a for a, m in zip(valid_actual, mask_non_empty) if m]
valid_pred_lgbm_pos = [p for p, m in zip(valid_pred_lgbm, mask_non_empty) if m]

lgbm_metrics_pos = mean_metrics(valid_actual_pos, valid_pred_lgbm_pos, k=5)
print("LightGBM one-vs-rest metrics (positive only):", lgbm_metrics_pos)

# MLflow логирование
log_experiment(
    "lgbm_ovr",
    {
        **{f"all_{k}": v for k, v in lgbm_metrics_all.items()},
        **{f"pos_{k}": v for k, v in lgbm_metrics_pos.items()},
    },
    {
        "model_type": "ovr",
        "n_models": len(models),
        "top_k": K,
    }
)

## Сравнение моделей

In [ ]:
def add_model_result(
    results_df: pd.DataFrame,
    model_name: str,
    metrics: dict,
    extra_info: dict | None = None,
    sort_by: str | None = None,
    ascending: bool = False,
) -> pd.DataFrame:
    """
    Добавляет результаты модели в таблицу сравнения.

    Parameters
    ----------
    results_df : pd.DataFrame
        Текущая таблица результатов.
    model_name : str
        Название модели.
    metrics : dict
        Словарь метрик, например:
        {"precision_at_5": 0.01, "recall_at_5": 0.03, "map_at_5": 0.02}
    extra_info : dict | None
        Дополнительная информация, например:
        {"split": "valid", "notes": "with prev features"}
    sort_by : str | None
        Название колонки для сортировки (например, "map_at_5").
    ascending : bool
        Направление сортировки (False — по убыванию, True — по возрастанию).

    Returns
    -------
    pd.DataFrame
        Обновлённая таблица результатов.
    """
    row = {"model": model_name}

    # добавляем метрики (с безопасным приведением типов)
    for k, v in metrics.items():
        try:
            row[k] = float(v)
        except (TypeError, ValueError):
            row[k] = v

    # добавляем доп. информацию (без перезаписи метрик)
    if extra_info is not None:
        for k, v in extra_info.items():
            if k not in row:
                row[k] = v
            else:
                row[f"extra_{k}"] = v

    row_df = pd.DataFrame([row])

    if results_df is None or results_df.empty:
        result = row_df
    else:
        result = pd.concat([results_df, row_df], ignore_index=True)

    # сортировка по выбранной метрике
    if sort_by is not None and sort_by in result.columns:
        result = result.sort_values(sort_by, ascending=ascending).reset_index(drop=True)

    return result

In [ ]:
comparison_df = pd.DataFrame()

baseline_metrics_pos = mean_metrics(valid_actual_pos, valid_pred_top_pop_pos, k=5)
lgbm_metrics_pos = mean_metrics(valid_actual_pos, valid_pred_lgbm_pos, k=5)

comparison_df = add_model_result(
    comparison_df,
    model_name="Top Popular",
    metrics=baseline_metrics,
    extra_info={"evaluation": "all_rows", "family": "baseline"},
)

comparison_df = add_model_result(
    comparison_df,
    model_name="LightGBM OVR",
    metrics=lgbm_metrics_all,
    extra_info={"evaluation": "all_rows", "family": "ovr"},
)

comparison_df = add_model_result(
    comparison_df,
    model_name="Top Popular",
    metrics=baseline_metrics_pos,
    extra_info={"evaluation": "positive_rows_only", "family": "baseline"},
)

comparison_df = add_model_result(
    comparison_df,
    model_name="LightGBM OVR",
    metrics=lgbm_metrics_pos,
    extra_info={"evaluation": "positive_rows_only", "family": "ovr"},
)

comparison_df = comparison_df.sort_values(
    ["evaluation", "map_at_5"], ascending=[True, False]
).reset_index(drop=True)

preferred_cols = [
    "evaluation",
    "family",
    "model",
    "precision_at_5",
    "recall_at_5",
    "map_at_5",
]

comparison_df = comparison_df[[c for c in preferred_cols if c in comparison_df.columns]]

numeric_cols = comparison_df.select_dtypes(include="number").columns
comparison_df[numeric_cols] = comparison_df[numeric_cols].round(4)

display(comparison_df)

## Вывод по экспериментам

В рамках эксперимента были сравнены два подхода: базовая модель **Top Popular** и модель **LightGBM (one-vs-rest)**.

Результаты показали, что:

* На полном наборе пользователей (**all_rows**) модель LightGBM уступает baseline:
  `map@5 = 0.0018` против `0.019` у Top Popular.
* На подмножестве пользователей с фактическими покупками (**positive_rows_only**) разрыв ещё более значительный:
  `map@5 = 0.06` против `0.63`.

Таким образом, модель LightGBM в текущей постановке задачи не смогла извлечь полезный сигнал и уступает простому популярностному подходу.

---

## Интерпретация результатов

Основные причины низкого качества модели:

1. **Сильный дисбаланс классов**
   Большинство наблюдений не содержит новых продуктов, что приводит к доминированию отрицательного класса и смещению модели в сторону предсказания нулей.

2. **Отсутствие этапа генерации кандидатов**
   Модель обучается предсказывать вероятность для всех продуктов сразу, включая нерелевантные, что значительно усложняет задачу.

3. **Неподходящая постановка задачи**
   One-vs-rest решает задачу бинарной классификации, тогда как задача рекомендаций по своей природе является задачей ранжирования.

4. **Ограниченный набор признаков**
   Используемые признаки недостаточно информативны для качественной персонализации.

---

## Вывод для дальнейшего моделирования

Полученные результаты показывают, что для повышения качества рекомендаций необходимо перейти к **двухэтапной архитектуре**:

1. Генерация кандидатов (например, с помощью ALS или Top Popular)
2. Ранжирование кандидатов с использованием модели машинного обучения (LightGBM)

Такой подход позволяет существенно сократить пространство поиска и сконцентрироваться на наиболее релевантных продуктах, что обычно приводит к значительному росту метрик качества.

---

## Итог

Проведённый эксперимент демонстрирует, что:

* простые модели могут быть сильным baseline,
* корректная постановка задачи (ranking vs classification) критически важна,
* для рекомендательных систем эффективнее использовать гибридные подходы.

## Логирование в MLflow

In [ ]:
import mlflow

mlflow.set_tracking_uri("http://127.0.0.1:5001")
mlflow.set_experiment("bank_product_recommendation")

with mlflow.start_run(run_name="lightgbm_one_vs_rest_v1"):

    # параметры
    mlflow.log_param("model_type", "lightgbm_one_vs_rest")
    mlflow.log_param("top_k", K)
    mlflow.log_param("num_features", len(feature_cols))
    mlflow.log_param("num_products", len(product_cols))

    # baseline метрики
    for metric_name, metric_value in baseline_metrics.items():
        mlflow.log_metric(f"baseline_{metric_name}", float(metric_value))

    # модельные метрики
    for metric_name, metric_value in lgbm_metrics_all.items():
        mlflow.log_metric(metric_name, float(metric_value))

print("MLflow logging completed")

## Сохранение модели

In [ ]:
MODELS_DIR = os.path.join(PROJECT_ROOT, "models")
os.makedirs(MODELS_DIR, exist_ok=True)

# pkl
joblib.dump(models, os.path.join(MODELS_DIR, "lgbm_one_vs_rest.pkl"))

# bin
for target_col, model in models.items():
    model.save_model(os.path.join(MODELS_DIR, f"{target_col}.bin"))

print("Модели сохранены")

In [ ]:
print("TRAIN target sum:", train_df[target_cols].sum().sum())
print("VALID target sum:", valid_df[target_cols].sum().sum())

valid_actual = valid_df[target_cols].apply(
    lambda row: [c.replace("target_", "") for c in target_cols if row[c] == 1],
    axis=1
).tolist()

num_non_empty_actual = sum(len(x) > 0 for x in valid_actual)
print("Non-empty actual in valid:", num_non_empty_actual)
print("Share:", round(num_non_empty_actual / len(valid_actual), 4))

print("\nTop valid targets:")
display(valid_df[target_cols].sum().sort_values(ascending=False).head(10))

print("\nScore stats:")
display(valid_scores.describe())

print("\nExamples:")
for i in range(5):
    print("ACTUAL:", valid_actual[i])
    print("PRED  :", valid_pred_lgb[i])
    print("-" * 40)

In [ ]:
# 1. actual
valid_actual = valid_df[target_cols].apply(
    lambda row: [c.replace("target_", "") for c in target_cols if row[c] == 1],
    axis=1
).tolist()

# 2. метрики по всем
print("Baseline metrics (all):", mean_metrics(valid_actual, valid_pred_top_pop, k=5))
print("LightGBM metrics (all):", mean_metrics(valid_actual, valid_pred_lgb, k=5))

# 3. метрики только по строкам с положительными событиями
mask_non_empty = [len(x) > 0 for x in valid_actual]

valid_actual_pos = [a for a, m in zip(valid_actual, mask_non_empty) if m]
valid_pred_top_pop_pos = [p for p, m in zip(valid_pred_top_pop, mask_non_empty) if m]
valid_pred_lgb_pos = [p for p, m in zip(valid_pred_lgb, mask_non_empty) if m]

print("Baseline metrics (positive rows only):", mean_metrics(valid_actual_pos, valid_pred_top_pop_pos, k=5))
print("LightGBM metrics (positive rows only):", mean_metrics(valid_actual_pos, valid_pred_lgb_pos, k=5))

# 4. сколько таких строк
print("Positive rows in valid:", sum(mask_non_empty), "out of", len(mask_non_empty))

# Обучение гибридной модели. Двухстадийный подход

## Собираем candidate set

In [ ]:
target_map = {c.replace("target_", ""): c for c in target_cols}


def get_combined_candidates(
    user_id,
    row,
    als_model,
    user2idx,
    user_item_matrix,
    idx2item,
    product_cols,
    popular_products,
    n_als=15,
    n_pop=10,
    total_candidates=20,
):
    als_candidates = recommend_als_candidates(
        user_id=user_id,
        row=row,
        als_model=als_model,
        user2idx=user2idx,
        user_item_matrix=user_item_matrix,
        idx2item=idx2item,
        product_cols=product_cols,
        n_candidates=n_als,
    )

    pop_candidates = get_top_popular_candidates(
        row=row,
        product_cols=product_cols,
        popular_products=popular_products,
        n_candidates=n_pop,
    )

    merged = {}

    for product, score in als_candidates:
        merged[product] = {
            "als_score": float(score),
            "candidate_source_als": 1,
            "candidate_source_pop": 0,
        }

    for product, _ in pop_candidates:
        if product not in merged:
            merged[product] = {
                "als_score": 0.0,
                "candidate_source_als": 0,
                "candidate_source_pop": 1,
            }
        else:
            merged[product]["candidate_source_pop"] = 1

    result = []
    for product, feats in merged.items():
        result.append({
            "candidate_product": product,
            "als_score": feats["als_score"],
            "candidate_source_als": feats["candidate_source_als"],
            "candidate_source_pop": feats["candidate_source_pop"],
        })

    result = sorted(
        result,
        key=lambda x: (x["candidate_source_als"], x["als_score"], x["candidate_source_pop"]),
        reverse=True,
    )

    return result[:total_candidates]


def build_candidate_dataset(
    base_df,
    product_cols,
    target_cols,
    als_model,
    user2idx,
    user_item_matrix,
    idx2item,
    popular_products,
    n_als=15,
    n_pop=10,
    total_candidates=20,
):
    rows = []

    target_map = {c.replace("target_", ""): c for c in target_cols}

    for _, row in base_df.iterrows():
        user_id = row["ncodpers"]

        candidates = get_combined_candidates(
            user_id=user_id,
            row=row,
            als_model=als_model,
            user2idx=user2idx,
            user_item_matrix=user_item_matrix,
            idx2item=idx2item,
            product_cols=product_cols,
            popular_products=popular_products,
            n_als=n_als,
            n_pop=n_pop,
            total_candidates=total_candidates,
        )

        for cand in candidates:
            product = cand["candidate_product"]
            target_col = target_map[product]

            rows.append({
                "ncodpers": user_id,
                "fecha_dato": row["fecha_dato"],
                "candidate_product": product,
                "label": int(row[target_col]),
                "als_score": cand["als_score"],
                "candidate_source_als": cand["candidate_source_als"],
                "candidate_source_pop": cand["candidate_source_pop"],
            })

    return pd.DataFrame(rows)

In [ ]:
train_candidates = build_candidate_dataset(
    base_df=train_df,
    product_cols=product_cols,
    target_cols=target_cols,
    als_model=als_model,
    user2idx=user2idx,
    user_item_matrix=user_item_matrix,
    idx2item=idx2item,
    popular_products=popular_products,
    n_als=15,
    n_pop=10,
    total_candidates=20,
)

valid_candidates = build_candidate_dataset(
    base_df=valid_df,
    product_cols=product_cols,
    target_cols=target_cols,
    als_model=als_model,
    user2idx=user2idx,
    user_item_matrix=user_item_matrix,
    idx2item=idx2item,
    popular_products=popular_products,
    n_als=15,
    n_pop=10,
    total_candidates=20,
)

print("train_candidates:", train_candidates.shape)
print("valid_candidates:", valid_candidates.shape)
display(train_candidates.head())

## Мержим user-level признаки

In [ ]:
train_user_features = train_df[["ncodpers", "fecha_dato"] + feature_cols].copy()
valid_user_features = valid_df[["ncodpers", "fecha_dato"] + feature_cols].copy()

train_rank_df = train_candidates.merge(
    train_user_features,
    on=["ncodpers", "fecha_dato"],
    how="left",
)

valid_rank_df = valid_candidates.merge(
    valid_user_features,
    on=["ncodpers", "fecha_dato"],
    how="left",
)

print("train_rank_df:", train_rank_df.shape)
print("valid_rank_df:", valid_rank_df.shape)

## Добавляем item-level признаки

In [ ]:
train_rank_df["product_popularity"] = (
    train_rank_df["candidate_product"].map(product_popularity_map).fillna(0.0)
)
valid_rank_df["product_popularity"] = (
    valid_rank_df["candidate_product"].map(product_popularity_map).fillna(0.0)
)

train_rank_df["product_prev_popularity"] = (
    train_rank_df["candidate_product"].map(product_prev_popularity_map).fillna(0.0)
)
valid_rank_df["product_prev_popularity"] = (
    valid_rank_df["candidate_product"].map(product_prev_popularity_map).fillna(0.0)
)

## Добавляем признак "был ли продукт раньше"

Он часто будет нулевой, но полезен как защитный сигнал.

In [ ]:
def add_prev_owned_candidate_feature(rank_df, product_cols):
    rank_df = rank_df.copy()
    rank_df["prev_owned_candidate"] = 0

    for p in product_cols:
        prev_col = f"prev_{p}"
        if prev_col in rank_df.columns:
            mask = rank_df["candidate_product"] == p
            rank_df.loc[mask, "prev_owned_candidate"] = rank_df.loc[mask, prev_col].fillna(0).astype(int)

    return rank_df


train_rank_df = add_prev_owned_candidate_feature(train_rank_df, product_cols)
valid_rank_df = add_prev_owned_candidate_feature(valid_rank_df, product_cols)

## Готовим признаки для ранжирования

In [ ]:
rank_cat_features = cat_features + ["candidate_product"]

rank_num_features = [
    c
    for c in train_rank_df.columns
    if c not in ["ncodpers", "fecha_dato", "label"]
    and c not in rank_cat_features
]

In [ ]:
# Приведение типов:
for col in rank_cat_features:
    train_rank_df[col] = train_rank_df[col].fillna("UNKNOWN").astype(str).astype("category")
    valid_rank_df[col] = valid_rank_df[col].fillna("UNKNOWN").astype(str).astype("category")

for col in rank_num_features:
    train_rank_df[col] = pd.to_numeric(train_rank_df[col], errors="coerce")
    valid_rank_df[col] = pd.to_numeric(valid_rank_df[col], errors="coerce")

    median_value = train_rank_df[col].median()
    train_rank_df[col] = train_rank_df[col].fillna(median_value)
    valid_rank_df[col] = valid_rank_df[col].fillna(median_value)

rank_feature_cols = rank_num_features + rank_cat_features

print("rank_cat_features:", rank_cat_features)
print("n rank_num_features:", len(rank_num_features))
print("n rank_feature_cols:", len(rank_feature_cols))

## Negative sampling для train

Иначе train будет очень перекошен.

In [ ]:
pos_train = train_rank_df[train_rank_df["label"] == 1].copy()
neg_train = train_rank_df[train_rank_df["label"] == 0].copy()

neg_ratio = 5
neg_train_sampled = neg_train.sample(
    n=min(len(neg_train), len(pos_train) * neg_ratio),
    random_state=42,
)

train_rank_sampled = pd.concat([pos_train, neg_train_sampled], axis=0)
train_rank_sampled = train_rank_sampled.sample(frac=1, random_state=42).reset_index(drop=True)

print("pos_train:", len(pos_train))
print("neg_train:", len(neg_train))
print("neg_train_sampled:", len(neg_train_sampled))
print("train_rank_sampled:", train_rank_sampled.shape)
print("positive rate:", round(train_rank_sampled["label"].mean(), 4))

## Обучаем один LightGBM reranker

In [ ]:
import lightgbm as lgb

X_train_rank = train_rank_sampled[rank_feature_cols].copy()
y_train_rank = train_rank_sampled["label"].astype(int)

X_valid_rank = valid_rank_df[rank_feature_cols].copy()
y_valid_rank = valid_rank_df["label"].astype(int)

train_dataset = lgb.Dataset(
    X_train_rank,
    label=y_train_rank,
    categorical_feature=rank_cat_features,
    free_raw_data=False,
)

valid_dataset = lgb.Dataset(
    X_valid_rank,
    label=y_valid_rank,
    categorical_feature=rank_cat_features,
    reference=train_dataset,
    free_raw_data=False,
)

rank_params = {
    "objective": "binary",
    "metric": ["binary_logloss", "auc"],
    "learning_rate": 0.05,
    "num_leaves": 63,
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq": 1,
    "min_data_in_leaf": 50,
    "verbosity": -1,
    "seed": 42,
    "is_unbalance": True,
}

rank_model = lgb.train(
    rank_params,
    train_dataset,
    valid_sets=[train_dataset, valid_dataset],
    valid_names=["train", "valid"],
    num_boost_round=500,
    callbacks=[
        lgb.early_stopping(50),
        lgb.log_evaluation(50),
    ],
)

## Предсказываем score для valid-кандидатов

In [ ]:
valid_rank_df = valid_rank_df.copy()
valid_rank_df["score"] = rank_model.predict(
    valid_rank_df[rank_feature_cols],
    num_iteration=rank_model.best_iteration,
)

display(valid_rank_df.head())

## Собираем top-k рекомендации

In [ ]:
def build_topk_predictions(rank_df, k=5):
    pred_dict = {}

    for user_id, group in rank_df.groupby("ncodpers"):
        group_sorted = group.sort_values("score", ascending=False)
        pred_dict[user_id] = group_sorted["candidate_product"].head(k).tolist()

    return pred_dict


valid_pred_hybrid_dict = build_topk_predictions(valid_rank_df, k=5)

## Actual для validation

In [ ]:
def extract_actual_products(row, target_cols):
    return [c.replace("target_", "") for c in target_cols if row[c] == 1]


valid_actual_dict = {}
for _, row in valid_df.iterrows():
    valid_actual_dict[row["ncodpers"]] = extract_actual_products(row, target_cols)

valid_actual_hybrid = []
valid_pred_hybrid = []

for user_id in valid_df["ncodpers"].tolist():
    valid_actual_hybrid.append(valid_actual_dict.get(user_id, []))
    valid_pred_hybrid.append(valid_pred_hybrid_dict.get(user_id, []))

## Считаем метрики hybrid-модели

In [ ]:
K = 5

hybrid_metrics_all = mean_metrics(valid_actual_hybrid, valid_pred_hybrid, k=K)
print("Hybrid metrics (all):", hybrid_metrics_all)

mask_non_empty_hybrid = [len(x) > 0 for x in valid_actual_hybrid]
valid_actual_hybrid_pos = [a for a, m in zip(valid_actual_hybrid, mask_non_empty_hybrid) if m]
valid_pred_hybrid_pos = [p for p, m in zip(valid_pred_hybrid, mask_non_empty_hybrid) if m]

hybrid_metrics_pos = mean_metrics(valid_actual_hybrid_pos, valid_pred_hybrid_pos, k=K)
print("Hybrid metrics (positive only):", hybrid_metrics_pos)

with mlflow.start_run(run_name="hybrid_als_lgbm_reranker"):

    mlflow.log_params({
        "model_type": "hybrid",
        "candidate_generator": "ALS + TopPopular",
        "reranker": "LightGBM",
        "top_k": K,
        "n_als_candidates": 15,
        "n_pop_candidates": 10,
        "total_candidates": 20,
        "negative_ratio": 5,
        "learning_rate": 0.05,
        "num_leaves": 63,
        "random_seed": 42,
        "n_train_rows": len(train_df),
        "n_valid_rows": len(valid_df),
        "train_candidates_rows": len(train_candidates),
        "valid_candidates_rows": len(valid_candidates),
        "n_rank_features": len(rank_feature_cols),
        "best_iteration": int(rank_model.best_iteration),
    })

    for metric_name, metric_value in hybrid_metrics_all.items():
        mlflow.log_metric(f"all_{metric_name}", float(metric_value))

    for metric_name, metric_value in hybrid_metrics_pos.items():
        mlflow.log_metric(f"pos_{metric_name}", float(metric_value))

    mlflow.lightgbm.log_model(rank_model, "model")

    mlflow.log_dict({"features": rank_feature_cols}, "features.json")

    mlflow.log_artifact("models/hybrid_rank_model.pkl")
    mlflow.log_artifact("models/hybrid_recommendations.parquet")
    mlflow.log_artifact("models/comparison_df.csv")

    mlflow.set_tag("stage", "final")
    mlflow.set_tag("model_family", "hybrid")

    print("MLflow hybrid run logged")

## Добавляем hybrid в общую таблицу сравнения

In [ ]:
comparison_df = add_model_result(
    comparison_df,
    model_name="ALS + LightGBM",
    metrics=hybrid_metrics_all,
    extra_info={"evaluation": "all_rows", "family": "hybrid"},
)

comparison_df = add_model_result(
    comparison_df,
    model_name="ALS + LightGBM",
    metrics=hybrid_metrics_pos,
    extra_info={"evaluation": "positive_rows_only", "family": "hybrid"},
)

comparison_df = comparison_df.sort_values(
    ["evaluation", "map_at_5"],
    ascending=[True, False],
).reset_index(drop=True)

preferred_cols = [
    "evaluation",
    "family",
    "model",
    "precision_at_5",
    "recall_at_5",
    "map_at_5",
]

comparison_df = comparison_df[[c for c in preferred_cols if c in comparison_df.columns]]

numeric_cols = comparison_df.select_dtypes(include="number").columns
comparison_df[numeric_cols] = comparison_df[numeric_cols].round(4)

display(comparison_df)

# Логируем финальные результаты

In [ ]:
# ================================
# 0. Импорты и подготовка
# ================================
import os
import json
import joblib
import mlflow
import mlflow.lightgbm
import pandas as pd

MODELS_DIR = "models"
os.makedirs(MODELS_DIR, exist_ok=True)

mlflow.set_experiment("bank_product_recommendation")


# ================================
# 1. Подготовка рекомендаций
# ================================
hybrid_recommendations_df = pd.DataFrame({
    "ncodpers": valid_df["ncodpers"].tolist(),
    "actual_products": valid_actual_hybrid,
    "predicted_products_top5": valid_pred_hybrid,
})

hybrid_recommendations_df["actual_count"] = hybrid_recommendations_df["actual_products"].apply(len)
hybrid_recommendations_df["pred_count"] = hybrid_recommendations_df["predicted_products_top5"].apply(len)


# ================================
# 2. Пути для сохранения
# ================================
rank_model_path = os.path.join(MODELS_DIR, "hybrid_rank_model.pkl")
rank_features_path = os.path.join(MODELS_DIR, "rank_feature_cols.pkl")
rank_cat_features_path = os.path.join(MODELS_DIR, "rank_cat_features.pkl")

als_model_path = os.path.join(MODELS_DIR, "als_model.pkl")
user2idx_path = os.path.join(MODELS_DIR, "user2idx.pkl")
item2idx_path = os.path.join(MODELS_DIR, "item2idx.pkl")
idx2item_path = os.path.join(MODELS_DIR, "idx2item.pkl")

popular_products_path = os.path.join(MODELS_DIR, "popular_products.pkl")
product_popularity_map_path = os.path.join(MODELS_DIR, "product_popularity_map.pkl")
product_prev_popularity_map_path = os.path.join(MODELS_DIR, "product_prev_popularity_map.pkl")

valid_rank_df_path = os.path.join(MODELS_DIR, "valid_rank_scored.parquet")
recommendations_path = os.path.join(MODELS_DIR, "hybrid_recommendations.parquet")
comparison_path = os.path.join(MODELS_DIR, "comparison_df.csv")
metrics_path = os.path.join(MODELS_DIR, "hybrid_metrics.json")
feature_importance_path = os.path.join(MODELS_DIR, "feature_importance.csv")


# ================================
# 3. Сохранение моделей и данных
# ================================
joblib.dump(rank_model, rank_model_path)
joblib.dump(rank_feature_cols, rank_features_path)
joblib.dump(rank_cat_features, rank_cat_features_path)

joblib.dump(als_model, als_model_path)
joblib.dump(user2idx, user2idx_path)
joblib.dump(item2idx, item2idx_path)
joblib.dump(idx2item, idx2item_path)

joblib.dump(popular_products, popular_products_path)
joblib.dump(product_popularity_map, product_popularity_map_path)
joblib.dump(product_prev_popularity_map, product_prev_popularity_map_path)

valid_rank_df.to_parquet(valid_rank_df_path, index=False)
hybrid_recommendations_df.to_parquet(recommendations_path, index=False)
comparison_df.to_csv(comparison_path, index=False)

with open(metrics_path, "w", encoding="utf-8") as f:
    json.dump(
        {
            "hybrid_metrics_all": hybrid_metrics_all,
            "hybrid_metrics_pos": hybrid_metrics_pos,
        },
        f,
        ensure_ascii=False,
        indent=2,
    )


# ================================
# 4. Feature importance
# ================================
feature_importance_df = pd.DataFrame({
    "feature": rank_feature_cols,
    "importance_gain": rank_model.feature_importance(importance_type="gain"),
    "importance_split": rank_model.feature_importance(importance_type="split"),
}).sort_values("importance_gain", ascending=False)

feature_importance_df.to_csv(feature_importance_path, index=False)


# ================================
# 5. MLflow логирование
# ================================
with mlflow.start_run(run_name="hybrid_als_lgbm_reranker"):

    mlflow.log_params({
        "model_type": "hybrid_recommender",
        "candidate_generator": "ALS + TopPopular",
        "reranker": "LightGBM",
        "n_als_candidates": 15,
        "n_pop_candidates": 10,
        "total_candidates": 20,
        "negative_ratio": 5,
        "learning_rate": 0.05,
        "num_leaves": 63,
        "feature_fraction": 0.8,
        "bagging_fraction": 0.8,
        "min_data_in_leaf": 50,
        "best_iteration": int(rank_model.best_iteration),
        "n_features": len(rank_feature_cols),
        "n_cat_features": len(rank_cat_features),
    })

    # метрики
    for k, v in hybrid_metrics_all.items():
        mlflow.log_metric(f"all_{k}", float(v))

    for k, v in hybrid_metrics_pos.items():
        mlflow.log_metric(f"pos_{k}", float(v))

    # модель
    mlflow.lightgbm.log_model(rank_model, artifact_path="model")

    # артефакты
    mlflow.log_artifact(rank_model_path)
    mlflow.log_artifact(rank_features_path)
    mlflow.log_artifact(rank_cat_features_path)

    mlflow.log_artifact(als_model_path)
    mlflow.log_artifact(user2idx_path)
    mlflow.log_artifact(item2idx_path)
    mlflow.log_artifact(idx2item_path)

    mlflow.log_artifact(popular_products_path)
    mlflow.log_artifact(product_popularity_map_path)
    mlflow.log_artifact(product_prev_popularity_map_path)

    mlflow.log_artifact(valid_rank_df_path)
    mlflow.log_artifact(recommendations_path)
    mlflow.log_artifact(comparison_path)
    mlflow.log_artifact(metrics_path)
    mlflow.log_artifact(feature_importance_path)

    print("MLflow run completed successfully.")


# ================================
# 6. Итог
# ================================
print("Hybrid model saved to:", rank_model_path)
print("Recommendations saved to:", recommendations_path)
print("Comparison table saved to:", comparison_path)